# Прогноз PJME_MW

Здесь оставил обычный порядок работы: загрузка данных, несколько графиков по сезонности, признаки с лагами, валидация по последним периодам и финальный файл для отправки.


In [ ]:
import sys
!{sys.executable} -m pip install lightgbm xgboost holidays --quiet


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import xgboost as xgb
import holidays
from sklearn.metrics import mean_absolute_percentage_error

sns.set_theme(style='whitegrid')


## Данные

Сначала смотрю границы train/test и общий вид ряда.


In [ ]:
train = pd.read_csv('data/train.csv', parse_dates=['Datetime'])
test = pd.read_csv('data/test.csv', parse_dates=['Datetime'])

train = train.sort_values('Datetime').reset_index(drop=True)
test = test.sort_values('Datetime').reset_index(drop=True)

print(train.shape, test.shape)
print(train.Datetime.min(), '->', train.Datetime.max())
print(test.Datetime.min(), '->', test.Datetime.max())
train.head()


## Общий вид ряда


In [ ]:
train['PJME_MW'].plot(figsize=(14, 3), title='полный ряд')
plt.ylabel('MW')
plt.show()


In [ ]:
# посмотреть суточный профиль по месяцам — летом и зимой явно разные пики
tmp = train.copy()
tmp['hour'] = tmp['Datetime'].dt.hour
tmp['month'] = tmp['Datetime'].dt.month

fig, ax = plt.subplots(figsize=(12, 4))
for month, g in tmp.groupby('month'):
    g.groupby('hour')['PJME_MW'].mean().plot(ax=ax, alpha=0.6, label=month)
ax.legend(title='месяц', ncol=6, fontsize=8)
ax.set_xlabel('час')
ax.set_ylabel('MW')
plt.show()


In [ ]:
# смотрю корреляцию с лагами — хочу понять какие реально нужны
corrs = {}
for lag in [1, 24, 48, 168, 336, 8736]:
    corrs[lag] = train['PJME_MW'].autocorr(lag=lag)
pd.Series(corrs, name='autocorr').round(3)


## Признаки

Использую календарные признаки, праздники, лаги, rolling-статистики и среднее по нескольким прошлым неделям.


In [ ]:
us_hols = holidays.US(years=range(2002, 2019))

lag_cols = [24, 48, 168, 8736, 17472]
roll_windows = [24, 168, 720]

def make_features(df):
    d = df['Datetime']
    df['hour']       = d.dt.hour
    df['dayofweek']  = d.dt.dayofweek
    df['month']      = d.dt.month
    df['year']       = d.dt.year
    df['dayofyear']  = d.dt.dayofyear
    df['quarter']    = d.dt.quarter
    df['weekofyear'] = d.dt.isocalendar().week.astype(int)
    df['is_weekend'] = (d.dt.dayofweek >= 5).astype(int)
    df['is_holiday'] = d.dt.date.map(lambda x: int(x in us_hols))
    df['hour_x_month'] = df['hour'] * df['month']
    df['hour_x_dow']   = df['hour'] * df['dayofweek']
    return df

full = pd.concat([train, test], sort=False).reset_index(drop=True)
full = make_features(full)

for lag in lag_cols:
    full[f'lag_{lag}'] = full['PJME_MW'].shift(lag)

s = full['PJME_MW'].shift(1)
for w in roll_windows:
    full[f'roll_mean_{w}'] = s.rolling(w).mean()
    full[f'roll_std_{w}']  = s.rolling(w).std()

full['lag_week_mean'] = (
    full['PJME_MW'].shift(168).rolling(4 * 168, min_periods=168).mean()
)

train_f = full[full['Datetime'] <= '2017-12-31 23:00:00'].dropna().reset_index(drop=True)
test_f  = full[full['Datetime'] >= '2018-01-01 00:00:00'].reset_index(drop=True)

features = [c for c in train_f.columns if c not in ('Datetime', 'PJME_MW')]

print(train_f.shape, test_f.shape, len(features))


## Валидация


In [ ]:
folds = [
    ('2016-01-01', '2016-06-30 23:00:00'),
    ('2016-07-01', '2016-12-31 23:00:00'),
    ('2017-01-01', '2017-12-31 23:00:00'),
]

params = {
    'n_estimators': 5000,
    'learning_rate': 0.02,
    'num_leaves': 127,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
}

cv_scores = []
for val_start, val_end in folds:
    tr = train_f[train_f['Datetime'] < val_start]
    va = train_f[(train_f['Datetime'] >= val_start) & (train_f['Datetime'] <= val_end)]

    m = lgb.LGBMRegressor(**params)
    m.fit(
        tr[features], tr['PJME_MW'],
        eval_set=[(va[features], va['PJME_MW'])],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(500)],
    )
    p = m.predict(va[features])
    score = mean_absolute_percentage_error(va['PJME_MW'], p) * 100
    cv_scores.append(score)
    print(f'{val_start[:7]} -> {val_end[:7]}   MAPE {score:.3f}%   trees {m.best_iteration_}')

print(f'
среднее {np.mean(cv_scores):.3f}%')


In [ ]:
# пробовал num_leaves=255 и learning_rate=0.01 — cv чуть хуже вышел, оставил 127/0.02
# params['num_leaves'] = 255
# params['learning_rate'] = 0.01


In [ ]:
fi = pd.Series(m.feature_importances_, index=features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
fi.head(20).plot(kind='bar', ax=ax)
ax.set_title('feature importance')
ax.set_ylabel('importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## Финальная модель


In [ ]:
best_iters = []
for val_start, val_end in folds:
    tr = train_f[train_f['Datetime'] < val_start]
    va = train_f[(train_f['Datetime'] >= val_start) & (train_f['Datetime'] <= val_end)]
    tmp = lgb.LGBMRegressor(**params)
    tmp.fit(tr[features], tr['PJME_MW'],
            eval_set=[(va[features], va['PJME_MW'])],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(9999)])
    best_iters.append(tmp.best_iteration_)

n_trees = int(np.mean(best_iters) * 1.05)
print('деревьев:', n_trees)

lgb_model = lgb.LGBMRegressor(**{**params, 'n_estimators': n_trees})
lgb_model.fit(train_f[features], train_f['PJME_MW'])

xgb_model = xgb.XGBRegressor(
    n_estimators=n_trees,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
)
xgb_model.fit(train_f[features], train_f['PJME_MW'], verbose=False)

print('готово')


## Прогноз test

На тесте значения идут подряд, поэтому лаги обновляю уже по мере прогноза.


In [ ]:
history = train[['Datetime', 'PJME_MW']].set_index('Datetime')['PJME_MW'].copy()

preds_lgb = []
preds_xgb = []

for _, row in test_f.iterrows():
    dt = row['Datetime']

    feats = {
        'hour':        dt.hour,
        'dayofweek':   dt.dayofweek,
        'month':       dt.month,
        'year':        dt.year,
        'dayofyear':   dt.dayofyear,
        'quarter':     dt.quarter,
        'weekofyear':  dt.isocalendar()[1],
        'is_weekend':  int(dt.dayofweek >= 5),
        'is_holiday':  int(dt.date() in us_hols),
        'hour_x_month': dt.hour * dt.month,
        'hour_x_dow':   dt.hour * dt.dayofweek,
    }

    for lag in lag_cols:
        feats[f'lag_{lag}'] = history.get(dt - pd.Timedelta(hours=lag), np.nan)

    recent = history[history.index < dt]
    for w in roll_windows:
        tail = recent.iloc[-w:]
        feats[f'roll_mean_{w}'] = tail.mean()
        feats[f'roll_std_{w}']  = tail.std()

    vals = [history.get(dt - pd.Timedelta(hours=168 * w), np.nan) for w in range(1, 5)]
    vals = [v for v in vals if not np.isnan(v)]
    feats['lag_week_mean'] = np.mean(vals) if vals else np.nan

    X = pd.DataFrame([feats])[features]
    p_lgb = lgb_model.predict(X)[0]
    p_xgb = xgb_model.predict(X)[0]
    preds_lgb.append(p_lgb)
    preds_xgb.append(p_xgb)

    history[dt] = (p_lgb + p_xgb) / 2

print('строк:', len(preds_lgb))


In [ ]:
preds_lgb = np.array(preds_lgb)
preds_xgb = np.array(preds_xgb)
ensemble = (preds_lgb + preds_xgb) / 2

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(test_f['Datetime'], ensemble, linewidth=0.8)
ax.set_title('Прогноз 2018')
ax.set_ylabel('MW')
plt.tight_layout()
plt.show()

print(ensemble.min(), ensemble.max(), ensemble.mean())


In [ ]:
submission = test[['Datetime']].copy()
submission['PJME_MW'] = ensemble

submission.to_csv('submission.csv', index=False)
submission.head()
